# Customer Segmentation — Clustering Analysis

This notebook combines the methodology presented in class with the extended clustering workflow from the reference project.

The workflow includes:

1. Loading the cleaned customer dataset  
2. Comparing scaling methods  
3. Evaluating K-Means  
4. Applying Ward Hierarchical Clustering  
5. Comparing K-Means and Ward solutions  
6. Selecting the final clustering solution  
7. Profiling and naming customer segments  
8. Visualizing clusters with PCA  
9. Exporting the clustered dataset  


## 1. Imports

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

import utils_clustering as utils

## 2. Load Cleaned Dataset

In [2]:
path = "/Users/franciscateixeira/Documents/ML/CostumerSegP_MLII/datasets/customer_info_clean.csv"

df = pd.read_csv(path, index_col="customer_id")

utils.validate_clustering_data(df)

Dataset shape: (32047, 24)
Missing values: 0
Duplicated rows: 0


,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,percentage_of_products_bought_promotion,customer_loyalty_flag,latitude,longitude,typical_hour_sin,typical_hour_cos,customer_age,education_level,is_male,tenure,total_children,total_spend,pct_spend_groceries,pct_spend_vegetables,pct_spend_nonalcohol_drinks,pct_spend_alcohol_drinks,pct_spend_meat,pct_spend_fish,pct_spend_hygiene,pct_spend_petfood,pct_spend_technology
customer_id,,,,,,,,,,,,,,,,,,,,,,,,
3,1,3.0,189.0,0.631599,1,38.794428,-9.215739,-0.366390,-5.381341e-01,56.0,1,0,6,2,18590.0,0.631038,0.020065,0.017375,0.009521,0.001506,0.011458,0.029693,0.020656,0.258687
4,0,2.0,130.0,0.149890,1,38.751711,-9.179611,-0.096472,-6.692130e-01,50.0,1,0,13,1,20233.0,0.676815,0.099442,0.026343,0.004695,0.002125,0.000741,0.092918,0.032867,0.064054
5,0,2.0,81.0,0.069126,0,38.780678,-9.160656,0.258819,-9.659258e-01,54.0,2,1,21,0,15549.0,0.797929,0.035694,0.006496,0.007589,0.081356,0.017557,0.032607,0.014277,0.006496
7,2,1.0,92.0,0.253609,1,38.739548,-9.148679,-1.000000,-2.220446e-16,43.0,0,1,5,0,14952.0,0.501137,0.005618,0.050629,0.075776,0.065008,0.072432,0.032437,0.012306,0.184658
8,3,1.0,6.0,0.186569,1,38.733071,-9.188188,-0.965926,-2.588190e-01,56.0,0,1,5,0,25797.0,0.356127,0.014730,0.022948,0.027833,0.041400,0.039346,0.011513,0.017095,0.469008


,count,mean,std,min,25%,50%,75%,max
number_complaints,32047.0,0.920897,0.895398,0.000000e+00,0.000000,1.000000e+00,1.000000,7.000000
distinct_stores_visited,32047.0,3.172185,1.672304,1.000000e+00,2.000000,3.000000e+00,4.000000,10.000000
lifetime_total_distinct_products,32047.0,149.216713,105.764038,0.000000e+00,67.000000,1.230000e+02,210.000000,600.000000
percentage_of_products_bought_promotion,32047.0,0.345946,0.264639,4.507807e-06,0.142758,2.602418e-01,0.496580,1.000000
customer_loyalty_flag,32047.0,0.604082,0.489055,0.000000e+00,0.000000,1.000000e+00,1.000000,1.000000
latitude,32047.0,38.749808,0.022482,3.868799e+01,38.734200,3.874832e+01,38.765891,38.823693
longitude,32047.0,-9.154518,0.028705,-9.232989e+00,-9.173839,-9.156666e+00,-9.139530,-9.035697
typical_hour_sin,32047.0,0.059553,0.708152,-1.000000e+00,-0.500000,1.249001e-16,0.866025,1.000000
typical_hour_cos,32047.0,-0.421295,0.554885,-1.000000e+00,-0.866025,-5.000000e-01,-0.258819,0.965926
customer_age,32047.0,55.424608,17.577420,2.400000e+01,40.000000,5.500000e+01,71.000000,86.000000


## 3. Create Scaled Versions

Clustering algorithms are distance-based, so scaling is essential.

We compare:

- StandardScaler
- MinMaxScaler
- RobustScaler

The unscaled dataset is not used as a final solution because the variables have very different scales.


In [3]:
scaled_versions, fitted_scalers = utils.create_scaled_versions(
    df,
    include_no_scaling=False
)

print(scaled_versions.keys())

dict_keys(['standard', 'minmax', 'robust'])


## 4. K-Means Evaluation

K-Means is evaluated for different values of `k`.

The following metrics are calculated:

- Inertia
- Silhouette Score
- Davies-Bouldin Score
- Calinski-Harabasz Score


In [4]:
kmeans_results = utils.evaluate_kmeans_for_scalers(
    scaled_versions,
    k_range=range(2, 11),
    random_state=42
)

K-Means evaluation with standard scaling


,k,inertia,silhouette,davies_bouldin,calinski_harabasz
0,2,693137.129768,0.159417,2.687776,3513.202976
1,3,638038.759227,0.118268,2.367821,3291.828199
2,4,595049.255702,0.110020,2.471851,3124.677077
3,5,563393.285553,0.119776,2.284497,2925.201504
4,6,539840.905964,0.112831,2.294144,2721.763057
5,7,520611.608609,0.098620,2.256028,2549.077327
6,8,502964.147201,0.105260,2.245689,2422.106007
7,9,487790.061851,0.106312,2.226949,2309.784735
8,10,476029.725391,0.111652,2.287485,2191.737644


K-Means evaluation with minmax scaling


,k,inertia,silhouette,davies_bouldin,calinski_harabasz
0,2,32678.640284,0.208049,1.871801,8505.722181
1,3,27837.768095,0.201331,1.697695,7778.419385
2,4,24665.124700,0.226343,1.713338,7226.332376
3,5,23393.889069,0.198580,1.873454,6149.380940
4,6,22151.598943,0.185984,2.036440,5554.613346
5,7,21381.104073,0.173405,2.078475,4987.934602
6,8,20545.095019,0.168134,2.123533,4635.449593
7,9,19822.697743,0.163269,2.078086,4349.648043
8,10,19116.636209,0.156952,2.058444,4140.501506


K-Means evaluation with robust scaling


,k,inertia,silhouette,davies_bouldin,calinski_harabasz
0,2,543173.881834,0.227107,1.457228,4320.800692
1,3,476443.401416,0.212062,1.931050,4706.950064
2,4,422126.419359,0.201421,1.735295,4916.003710
3,5,391850.140307,0.195690,1.666281,4590.687199
4,6,370789.487416,0.140687,2.003140,4245.009399
5,7,356006.504307,0.136169,1.964240,3906.028964
6,8,341099.417875,0.142757,1.968923,3694.264125
7,9,329832.224838,0.128168,1.953776,3479.601871
8,10,319962.226544,0.134670,1.923746,3298.095544


## 5. Elbow Curve and Silhouette Score

In [ ]:
utils.plot_kmeans_metrics(kmeans_results)

## 6. Ward Hierarchical Clustering — Dendrogram

Ward linkage is used as the hierarchical clustering method because it minimizes within-cluster variance.

A sample is used for the dendrogram to avoid excessive computation and unreadable plots.


In [ ]:
X_standard = scaled_versions["standard"]

linked = utils.plot_ward_dendrogram(
    X_standard,
    sample_size=3000,
    random_state=42,
    truncate_level=5
)

## 7. Choose Final Solution

Adjust `final_scaler_name` and `best_k` after inspecting:

- Elbow curve
- Silhouette score
- Dendrogram
- Business interpretability


In [ ]:
final_scaler_name = "standard"
best_k = 5

X_final = scaled_versions[final_scaler_name]

print(f"Final scaler: {final_scaler_name}")
print(f"Final number of clusters: {best_k}")

## 8. Fit Final K-Means Solution

In [ ]:
kmeans_final, kmeans_labels = utils.fit_kmeans_final(
    X_final,
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

df_clustered = df.copy()
df_clustered["cluster_kmeans"] = kmeans_labels

utils.plot_cluster_sizes(df_clustered, "cluster_kmeans")

## 9. Fit Ward Solution with the Same Number of Clusters

In [ ]:
ward_final, ward_labels = utils.fit_ward_clustering(
    X_final,
    n_clusters=best_k
)

df_clustered["cluster_ward"] = ward_labels

utils.plot_cluster_sizes(df_clustered, "cluster_ward")

## 10. Compare K-Means vs Ward

This comparison checks whether both clustering approaches identify similar customer structures.


In [ ]:
comparison = utils.compare_cluster_solutions(
    df_clustered["cluster_kmeans"],
    df_clustered["cluster_ward"],
    name_1="K-Means",
    name_2="Ward"
)

## 11. Choose Final Cluster Column

K-Means is usually easier to explain and reproduce.

Ward is used as a validation method. If Ward gives more interpretable segments, change `final_cluster_col` to `"cluster_ward"`.


In [ ]:
final_cluster_col = "cluster_kmeans"

df_clustered["final_cluster"] = df_clustered[final_cluster_col]

## 12. Cluster Profiling

Each cluster is profiled by comparing its feature averages against the global average.

Interpretation:

- Values above 1 indicate that the cluster is above the global average.
- Values below 1 indicate that the cluster is below the global average.


In [ ]:
original_features = df.columns.tolist()

cluster_means = utils.calculate_cluster_means(
    df_clustered,
    "final_cluster"
)

display(cluster_means)

cluster_profile = utils.calculate_cluster_profile(
    df_clustered,
    "final_cluster",
    original_features
)

display(cluster_profile.T)

utils.plot_cluster_profile_heatmap(cluster_profile)

## 13. Top Drivers per Cluster

In [ ]:
for cluster_id in sorted(df_clustered["final_cluster"].unique()):
    print(f"\nCluster {cluster_id} — Top differentiating features")
    display(
        utils.show_top_cluster_drivers(
            cluster_profile,
            cluster_id,
            top_n=10
        )
    )

## 14. Name Clusters

Update the mapping below after analyzing the cluster profiles.

Use professional names, such as:

- High Value Customers
- Promotion-Sensitive Customers
- Technology-Oriented Customers
- Family-Oriented Customers
- Pet-Oriented Customers
- Service-Sensitive Customers
- Low Engagement Customers
- Loyal Premium Customers


In [ ]:
cluster_names = {
    0: "Segment 0",
    1: "Segment 1",
    2: "Segment 2",
    3: "Segment 3",
    4: "Segment 4"
}

df_clustered["cluster_name"] = df_clustered["final_cluster"].map(cluster_names)

display(df_clustered["cluster_name"].value_counts())

## 15. PCA Visualization

In [ ]:
pca_df, pca_model = utils.plot_pca_clusters(
    X_final,
    df_clustered["final_cluster"],
    title="Customer Segments - PCA Projection"
)

## 16. Optional UMAP Visualization

Only run this cell if `umap-learn` is installed.

If it is not installed, PCA is enough for the report.


In [ ]:
# umap_df, umap_model = utils.plot_umap_clusters(
#     X_final,
#     df_clustered["final_cluster"],
#     title="Customer Segments - UMAP Projection"
# )

## 17. Export Final Clustered Dataset

In [ ]:
output_path = "/Users/franciscateixeira/Documents/ML/CostumerSegP_MLII/datasets/customer_info_clustered.csv"

utils.export_clustered_dataset(
    df_clustered,
    output_path
)

## 18. Methodology Text for Report

In [ ]:
methodology_text = """
After preprocessing, the cleaned customer dataset was used for clustering.
Since clustering algorithms are distance-based, different scaling methods were
compared: StandardScaler, MinMaxScaler and RobustScaler.

K-Means was evaluated for different numbers of clusters using the elbow method,
based on inertia, and the silhouette score. Additional validation metrics such
as Davies-Bouldin and Calinski-Harabasz were also calculated.

Hierarchical clustering using Ward linkage was applied as a complementary
method. A dendrogram was generated on a sample of the data to inspect the
hierarchical structure, and the final Ward solution was compared against the
K-Means solution.

The final clustering solution was selected based on quantitative metrics,
visual inspection and business interpretability. The resulting clusters were
profiled by comparing each cluster's feature averages against the global
average, allowing the segments to be assigned meaningful business names.
"""

print(methodology_text)